# 一键流程：PDF 文件夹 → 入库 → 发布 → 回答问题集

从上到下依次执行即可：对一个文件夹里的每份 PDF 依次做 ingest → requalify → qualify → index → publish → tree，
然后（可选）用同一套 `document-catalog` 链路在进程内回答一份问题集，并写出 `report.json` / `report.md`（固定写到项目根的 `data/reports/<题集文件名>/`）。
已处理过的 PDF 重跑只回放缓存，不再调用模型。

## 前置条件

1. 项目根 `.env` 按 `.env.example` 填好（`.env` 固定在项目根，与 notebook 的工作目录无关，不提交）：
   - LLM：`OPENAI_API_KEY` / `OPENAI_BASE_URL` / `OPENAI_MODEL`（别名 `APP_LLM_API_KEY` / `APP_LLM_BASE_URL` / `APP_LLM_MODEL`）；
   - Embedding：最简单是设 `OPENAI_EMBEDDING_MODEL`（与 LLM 同一个 `OPENAI_BASE_URL` 网关、同一个 key）；独立 loopback 服务才用 `APP_EMBEDDING_*`；
   - Rerank：`APP_RERANK_*`（本地模型，loopback 地址；没有网关回退）；
   - 输入位置：`NB_PDF_DIR`（必填，只读）、`NB_QUESTIONS_PATH`（别名 `DATASET_PATH`，可选）。输出位置不可配置：本 notebook **不使用** `NB_REPORT_DIR`（该变量只对 CLI / 直接调用 `run_folder_pipeline` 有效），见下面「运行产物位置」。
   - 口令加密的 PDF：`PDF_INGEST_PASSWORD`（不带 `APP_` 前缀；需要打开口令而它未设置或不对时，该 PDF 以 `failed` 结束、进度输出里直接写明原因；未加密的 PDF 不受影响）。
2. 用到本地模型（独立 embedding 服务或 rerank）时，**本地模型隧道要先起**（本流程自己不会启动它）：
   `.venv/bin/python scripts/enterprise_pdf_rag/local_model_tunnel.py start`
3. 用项目的 `.venv` 作为 kernel（已 `pip install -e .`）。

本 notebook 不含任何密钥，也不向 `os.environ` 写入配置。

## 只跑前 N 道题

配置 cell 里的 `MAX_QUESTIONS` 默认是 `10`：只回答题集的前 10 道题（按题集原顺序），先确认整条链路能跑通；改成 `None` 回答全部。它只限制回答，**不限制入库**：文件夹里的 PDF 仍会全部入库。

## 运行产物位置

所有运行产物都在项目根的 `data/` 下，**绝不写 PDF 源目录**（它按只读对待）；下列目录不存在时自动创建：

```text
<项目根>/data/
├── ingestion/                  # 入库产物：各文档的 source / processing store、model-cache、answers-audit.sqlite（APP_INGESTION_DIR 可改，但必须仍在 data/ 之内）
├── reports/<题集文件名>/        # report.json / report.md，每次运行都写（没有题集时目录名为 run-folder）；有题集时另有 answers.csv（question / expected / answer 三列）
```

配置 cell 之后有一格「写入目录护栏」：`APP_INGESTION_DIR`、报告目录解析后只要不在 `data/` 之内、或落在 PDF 源目录之内（或就是它），就在任何写入发生之前抛异常，Run All 停在那里。

## 诊断格（出问题时把输出发回）

- `version-check`（导入诊断之后）：打印当前代码的 git commit、kernel 实际加载的硬链接回退模块、`ragspine` 版本、已有 `report.json` 的修改时间（分清新旧报告）与平台信息；任何一步失败只打印原因。
- `fs-selfcheck`（写入目录护栏之后、主运行之前）：在 `data/ingestion/_selfcheck_*` 临时目录里逐项试入库会用到的文件操作（硬链接、`os.replace`、独占创建、追加写、sqlite 等）并调用生产函数 `link_new_file`，最后打印中文结论并清理；对 PDF 源目录只读第一份 PDF，不写入。入库报 `Operation not permitted` 或结果不对时，请把这两格的输出发回。
- `llm-selfcheck`（`fs-selfcheck` 之后、主运行之前）：用 ragspine 自己的 `JsonCompletionClient` 取得入库 text / vision、tree 建树、回答阶段四种真实请求体（只捕获、不发送），各向 LLM 端点发 1 次极小的合成请求；只有被拒的形态才逐项二分（`response_format`、schema 约束关键字、`$defs` / `anyOf`、`strict`、`temperature`、`max_completion_tokens`、`stream`、`seed`、`image_url`），并打印「结论」。每个请求只发固定的合成文本，不写磁盘、不打印 key。入库阶段形态被拒或连不上端点时，打印完诊断后抛异常停在这一格（否则只会得到 `nothing_to_index` 和失败缓存），请把本格输出整段发回；回答 / tree 形态被拒只警告。例外：根因是 `temperature` / `seed` 这类采样参数、且去掉它后入库形态通过时，只提示「管线会自动去掉它重发」、不阻断（ADR 0021；想省掉每次运行的一次被拒探测，可在 `.env` 设 `OPENAI_TEMPERATURE=omit`）。配置格的 `LLM_SELFCHECK = False` 可跳过本格。

## 测试数据

默认的合成 PDF 目录 `data/fixtures/pdf` 被 `.gitignore` 的 `data/*` 忽略，不随仓库同步；机器上没有时用以下命令生成（需要 dev 依赖 reportlab）：

```bash
.venv/bin/python scripts/make_fixtures_pdf.py
```

## 运行时行为

- 缺少必需配置时会在花任何 LLM 预算之前抛 `PreflightError` 并指出缺哪些变量。
- 开跑前会向 embedding 服务探测一次（`embed_query("preflight")`），用来确认网关或隧道已通。
- 只有题集里有 `rerank: true` 的用例时才需要 `APP_RERANK_*` 配置。

In [ ]:
# 安装本仓库（editable）及一键流程需要的 extras：pdf = pdfspine，service = 进程内评测用的 fastapi/httpx。
# 已经装过的环境可以跳过这一格。Databricks 上执行完要重启解释器：dbutils.library.restartPython()
%pip install -q -e "..[pdf,service]"

In [ ]:
import os

# 只在 Databricks 上重启 Python（让上一格 %pip 装的包生效）；其它环境什么都不做
# 本格是"notebook 不直接读环境变量"规则的例外：它在 import ragspine 之前执行，拿不到 get_settings()，只能直接读环境变量做平台探测
if os.environ.get("DATABRICKS_RUNTIME_VERSION") and "dbutils" in globals():
    dbutils.library.restartPython()  # noqa: F821 - dbutils 由 Databricks 注入
else:
    print("非 Databricks 环境，跳过")

In [ ]:
# editable 安装靠 .pth 把 <repo>/src 加进 sys.path；Databricks 上该 .pth 不生效时，
# site-packages/ragspine/ 只剩 _llms/（无 __init__.py），import ragspine 会命中这个
# namespace package，于是找不到 ragspine.common。这里显式把 src 放到 sys.path 最前
# （必须在 restart 之后）；.pth 已生效的环境里这只是无害的重复。
import sys
from pathlib import Path

_src = Path.cwd().parent / "src"
if (_src / "ragspine" / "__init__.py").is_file() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))
print("src 路径 =", _src, "| 已在 sys.path:", str(_src) in sys.path)

In [ ]:
# 导入诊断：确认 import ragspine 实际命中的是哪份代码（restart 之后执行）
# - __file__ 为 None            → 命中了同名目录形成的 namespace package，看 __path__ 是谁在遮蔽（见上一格 src-path）
# - __file__ 在 site-packages 下 → 装的是 PyPI 发行版（0.17.2 及更早没有 common/evidence/configs.py）
# - __file__ 指向 .../src/ragspine/__init__.py → 上面的 editable 安装已生效
import importlib.metadata as md
import importlib.util
import sys

try:
    import ragspine
except ImportError as exc:
    ragspine = None
    print("import ragspine 失败 =", repr(exc))

if ragspine is not None:
    print("ragspine.__file__ =", ragspine.__file__)
    print("ragspine.__path__ =", list(getattr(ragspine, "__path__", [])))
    for mod in ("ragspine.common", "ragspine.common.evidence", "ragspine.common.evidence.configs"):
        try:
            found = importlib.util.find_spec(mod) is not None
        except ImportError:
            found = False
        print(f"{mod}: {'OK' if found else '找不到'}")
for dist in ("ragspine", "rag-spine"):
    try:
        print(f"{dist} 版本 =", md.version(dist))
    except md.PackageNotFoundError:
        print(f"{dist} 未安装")
print("sys.path 前 8 项 =", sys.path[:8])

In [ ]:
# 版本检查（诊断，只打印、不抛异常）：确认 kernel 里跑的是不是最新代码，失败时把本格输出发回
import importlib
import inspect
import os
import platform
import subprocess
import sys
from datetime import datetime

try:
    from ragspine.common.evidence.configs import ROOT_DIR as _ROOT
except Exception as exc:  # noqa: BLE001 - 诊断格：任何失败只打印
    _ROOT = None
    print("无法确定项目根：", repr(exc))

if _ROOT is not None:
    try:
        _git = subprocess.run(
            ["git", "-C", str(_ROOT), "log", "-1", "--format=%h %s"],
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
        print(
            "当前代码 commit :",
            _git.stdout.strip()
            if _git.returncode == 0
            else f"无法取得 commit：git 返回 {_git.returncode}：{_git.stderr.strip()}",
        )
    except (OSError, subprocess.SubprocessError) as exc:
        print(f"当前代码 commit : 无法取得 commit：{type(exc).__name__}: {exc}")

try:
    _placement = importlib.import_module("ragspine.common.evidence.file_placement")
    print("硬链接回退模块  :", _placement.__file__)
    _store = importlib.import_module("enterprise_pdf_rag.adapters.document_store")
    print(
        "document_store 引用 link_new_file:",
        "是"
        if "link_new_file" in inspect.getsource(_store)
        else "否（新旧代码混用，请重启 kernel）",
    )
except ImportError as exc:
    print("修复未加载：代码未更新或未重启 kernel（", repr(exc), "）")
except Exception as exc:  # noqa: BLE001
    print("检查硬链接回退失败：", repr(exc))

try:
    import ragspine

    print("ragspine 版本   :", getattr(ragspine, "__version__", "未知"))
except Exception as exc:  # noqa: BLE001
    print("ragspine 版本   : 无法取得：", repr(exc))

if _ROOT is not None:
    try:
        _reports = sorted((_ROOT / "data" / "reports").rglob("report.json"))
        for _report in _reports:
            print(
                "已有报告        :",
                _report,
                "修改于",
                datetime.fromtimestamp(_report.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
            )
        if not _reports:
            print("已有报告        : 尚无报告")
    except OSError as exc:
        print("已有报告        : 无法列出：", repr(exc))

print("平台            :", platform.platform(), "| Python", sys.version.split()[0])
print("Databricks      :", os.environ.get("DATABRICKS_RUNTIME_VERSION") or "否")

In [ ]:
from ragspine.common.evidence.configs import get_settings
from ragspine.common.evidence.configs import ROOT_DIR

settings = get_settings()

# ---- 来自项目根 .env（或真实环境变量）：未设置时为 None ----
PDF_DIR = settings.pdf_source_dir  # NB_PDF_DIR
QUESTIONS = settings.questions_path  # NB_QUESTIONS_PATH：不设则只入库、不评测
# 报告目录固定为 <项目根>/data/reports/<题集文件名>/（没有题集时为 run-folder），每次运行都写 report.json / report.md；notebook 不使用报告目录的环境变量配置
REPORT_DIR = ROOT_DIR / "data" / "reports" / ("run-folder" if QUESTIONS is None else QUESTIONS.stem)
INGESTION_ROOT = settings.ingestion_root  # APP_INGESTION_DIR，默认 <APP_DATA_DIR>/ingestion = <项目根>/data/ingestion；必须在 data/ 之内（见下一格护栏）

# ---- 可在这里覆盖的调用参数 ----
MAX_LIVE_CALLS_PER_PDF = 100  # 每份 PDF 的 ingest 真实模型调用预算（0-200）；0 = 只回放缓存
MAX_LIVE_CALLS_TOTAL = None  # ingest + tree + 回答共用的总预算；None = 不设总预算
BUILD_TREE = True  # 发布后为每份文档建 document tree（路由用，会多花调用）
REQUALIFY = True  # 重新资格化图表等视觉对象（图表要靠它才进索引）
MAX_QUESTIONS = 10  # 只回答题集的前 N 道题，用来先确认整条链路能跑通；None = 回答全部（不影响入库：文件夹里的 PDF 仍会全部入库）
LLM_SELFCHECK = True  # 主运行之前先对 LLM 端点做一次请求体自检（约 4 次极小请求）；入库形态被拒或连不上就抛异常停在 llm-selfcheck 格；False = 跳过

print("PDF 源目录 :", PDF_DIR)
print("题集       :", QUESTIONS)
print("报告目录   :", REPORT_DIR)
print("ingestion  :", INGESTION_ROOT)
print("题数上限   :", "不限（回答题集全部）" if MAX_QUESTIONS is None else f"只回答题集前 {MAX_QUESTIONS} 道（MAX_QUESTIONS）")

In [ ]:
# ───────────── 写入目录护栏（必须在主运行之前）─────────────
# 本 notebook 的所有运行产物只许落在 <项目根>/data/ 下，且绝不写 PDF 源目录（它是只读的）。
# 任何一个写入目录不满足就在写入发生之前抛异常，让 Run All 停在这里；目录不存在由管线自动创建。
from pathlib import Path

DATA_DIR = (ROOT_DIR / "data").resolve()


def check_write_dir(path, label, variable):
    """写入目录合规返回 None，否则返回简体中文原因。"""
    target = Path(path).expanduser().resolve()
    if not target.is_relative_to(DATA_DIR):
        return f"{label} {target}（由 {variable} 决定）不在项目根的 data/ 目录（{DATA_DIR}）之内：所有运行产物只许写到 data/ 下"
    if PDF_DIR is not None and target.is_relative_to(Path(PDF_DIR).expanduser().resolve()):
        return f"{label} {target}（由 {variable} 决定）位于 PDF 源目录 {Path(PDF_DIR).expanduser().resolve()}（NB_PDF_DIR）之内或就是它：PDF 目录是只读的，不能往里写任何东西"
    return None


for _label, _path, _variable in (
    ("ingestion 目录", INGESTION_ROOT, "APP_INGESTION_DIR / APP_DATA_DIR"),
    ("报告目录", REPORT_DIR, "notebook 固定的 data/reports/<题集文件名>"),
):
    _problem = None if _path is None else check_write_dir(_path, _label, _variable)
    if _problem:
        raise RuntimeError("写入目录护栏拒绝运行（尚未写入任何文件）：" + _problem)
print("写入目录护栏通过：ingestion / 报告目录都在", DATA_DIR, "之内，且不在 PDF 源目录之内")

In [ ]:
# ───────────── 文件系统自检（诊断，不是护栏；失败只打印、不中断 Run All）─────────────
# 在 INGESTION_ROOT 下的临时子目录里逐项试一遍入库会用到的文件操作，并调用真正的生产函数 link_new_file；
# 结束后清理临时目录。对 PDF 源目录只做第一份 PDF 的 stat 与 read_bytes，绝不写入。失败时把本格输出发回。
import os
import shutil
import sqlite3
import tempfile
import uuid
from pathlib import Path


def _selfcheck():
    ok = {}

    def step(key, name, fn):
        try:
            out = fn()
            print(f"[成功] {name}" + (f"  -> {out}" if out else ""))
            ok[key] = True
        except BaseException as exc:  # noqa: BLE001 - 自检要把任何失败如实打印
            print(
                f"[失败] {name}\n       {type(exc).__name__}: {exc} (errno={getattr(exc, 'errno', None)})"
            )
            ok[key] = False
        return ok[key]

    def expect_exists(action):
        try:
            action()
        except FileExistsError:
            return "FileExistsError（正常）"
        raise RuntimeError("目标已存在却没抛 FileExistsError")

    first = (
        None
        if PDF_DIR is None
        else next(iter(sorted(Path(PDF_DIR).expanduser().rglob("*.pdf"))), None)
    )
    if first is None:
        print("PDF 源目录没有可读的 PDF（或未设置 NB_PDF_DIR），跳过源 PDF 的读取检查")
    else:
        step("pdf_stat", "stat 第一份 PDF（只读）", lambda: f"{first.stat().st_size} 字节")
        step("pdf_read", "read_bytes 第一份 PDF（只读）", lambda: f"{len(first.read_bytes())} 字节")

    work = Path(INGESTION_ROOT).expanduser() / f"_selfcheck_{uuid.uuid4().hex[:8]}"
    try:
        if not (
            step("mkdir", "mkdir(parents=True) 临时子目录", lambda: work.mkdir(parents=True))
            or work.is_dir()
        ):
            return ok
        tmp = {}

        def write_tmp():
            with tempfile.NamedTemporaryFile(dir=work, delete=False) as handle:
                tmp["path"] = Path(handle.name)
                handle.write(b"hello")
                handle.flush()
                os.fsync(handle.fileno())
            return tmp["path"].name

        if step("tmpwrite", "NamedTemporaryFile 写入 + fsync + close", write_tmp):
            final = work / "final"
            step(
                "replace_new",
                "os.replace 到新名（目标不存在）",
                lambda: os.replace(tmp["path"], final),
            )
            second = work / "second"
            second.write_bytes(b"world")
            step("replace_over", "os.replace 覆盖已存在目标", lambda: os.replace(second, final))
            if final.exists():
                link = work / "linked"
                step(
                    "link",
                    "os.link 硬链接（管线首选的“不存在才创建”）",
                    lambda: os.link(final, link),
                )
                if ok["link"]:
                    step(
                        "link_again",
                        "os.link 目标已存在应抛 FileExistsError",
                        lambda: expect_exists(lambda: os.link(final, link)),
                    )
            claim = work / "x.claim"
            step(
                "excl",
                "os.open(O_CREAT|O_EXCL) 独占创建",
                lambda: os.close(os.open(claim, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)),
            )
            step(
                "excl_again",
                "O_EXCL 对已存在文件应抛 FileExistsError",
                lambda: expect_exists(
                    lambda: os.open(claim, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
                ),
            )
            if os.name != "nt":

                def fsync_dir():
                    descriptor = os.open(work, os.O_RDONLY)
                    try:
                        os.fsync(descriptor)
                    finally:
                        os.close(descriptor)

                step("dir_fsync", "目录 fsync（失败不影响入库）", fsync_dir)
        whole = work / "whole.bin"

        def overwrite():
            whole.write_bytes(b"one")
            whole.write_bytes(b"two")

        step("overwrite", "整文件覆盖写（write_bytes 两次）", overwrite)

        def append():
            with open(work / "append.bin", "ab") as handle:
                handle.write(b"a")
            with open(work / "append.bin", "ab") as handle:
                handle.write(b"b")

        step("append", "追加写 open('ab')", append)

        def sqlite_wal():
            con = sqlite3.connect(work / "t.sqlite")
            try:
                mode = con.execute("PRAGMA journal_mode=WAL").fetchone()[0]
                con.execute("CREATE TABLE t(x)")
                con.execute("INSERT INTO t VALUES (1)")
                con.commit()
                return f"journal_mode={mode}"
            finally:
                con.close()

        step("sqlite", "sqlite 建库 + WAL + 写一行", sqlite_wal)

        try:
            from ragspine.common.evidence.file_placement import link_new_file
        except ImportError as exc:
            print(
                "[跳过] 生产函数 link_new_file：修复未加载（代码未更新或未重启 kernel）", repr(exc)
            )
        else:
            place = work / "placed"
            part_a, part_b = work / "part-a", work / "part-b"
            part_a.write_bytes(b"same")
            part_b.write_bytes(b"same")
            step("prod_new", "link_new_file 放置新文件", lambda: link_new_file(part_a, place))
            step(
                "prod_exists",
                "link_new_file 目标已存在应抛 FileExistsError",
                lambda: expect_exists(lambda: link_new_file(part_b, place)),
            )
    finally:
        try:
            shutil.rmtree(work)
            print("[清理] 已删除", work)
        except BaseException as exc:  # noqa: BLE001
            print("[清理失败]（仅提示）", type(exc).__name__, exc)
    return ok


try:
    _ok = _selfcheck()
except BaseException as exc:  # noqa: BLE001 - 诊断不得中断 Run All
    _ok = {}
    print("自检本身出现意外异常（已忽略，继续）：", type(exc).__name__, exc)

if (
    not _ok.get("mkdir")
    or not _ok.get("tmpwrite")
    or not (_ok.get("replace_new") and _ok.get("replace_over"))
):
    print("结论：os.replace 也不可用（或基础读写失败）→ 入库会失败，请把本格输出发回")
elif _ok.get("link") is False:
    print("结论：硬链接不可用，但回退所需的 os.replace 可用 → 入库应能正常进行")
else:
    print("结论：全部可用")
if _ok.get("prod_new") is False or _ok.get("prod_exists") is False:
    print("提示：生产函数 link_new_file 未通过，入库可能失败，请把本格输出发回")
if _ok.get("append") is False:
    print("提示：追加写不可用；本 notebook 的结果文件 answers.csv 是整文件写，不受影响")
if _ok.get("sqlite") is False:
    print("提示：sqlite 不可用；审计库写不了，回答原文直接记录在结果里（answers.csv），不受影响")

In [ ]:
# ───────────── LLM 请求自检（入库就 HTTP 400 时，把本格输出整段发回）─────────────
# 用 ragspine 自己的 JsonCompletionClient 取得管线真实会发的四种请求体（只捕获、不发送）：
#   入库 text（页元数据）/ 入库 vision（版面，带一张极小的合成 PNG）/ tree 建树 / 回答阶段（带 seed）；
# 再用标准库 http.client 按相同 URL / header 形态各发 1 次，读取服务端错误正文；
# 只有被拒的形态才逐项二分定位到具体字段 / schema 关键字。内容全是固定的极小合成文本，不含任何文档内容。
# 不写磁盘（缓存目录是用完即删的临时目录）、不写环境变量、不打印 key / Authorization。
# 入库阶段的形态被拒（或连不上）时，打印完全部诊断后抛异常，让 Run All 停在这一格；
# 唯一的例外：根因是管线能自动去掉重发的采样参数（temperature / seed，与实现共用同一白名单），
# 且去掉它后入库形态通过——这时只提示、不阻断（ADR 0021）。
# 配置格里的 LLM_SELFCHECK = False 可跳过本格。
import copy
import json
import time
import zlib
from http.client import HTTPException, HTTPSConnection
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.parse import urlsplit

from ragspine.common.evidence.configs import get_settings
from ragspine.common.evidence.providers.json_completion import (
    DEGRADABLE_SAMPLING_PARAMETERS,
    JsonCompletionClient,
)
from ragspine.common.evidence.providers.providers import (
    ProviderConfigurationError,
    ProviderRequestError,
    load_llm_config,
)

_SC_TIMEOUT_S = 30.0
_SC_BODY_CHARS = 300
_SC_CONSTRAINT_CLASSES = (
    ("长度 minLength/maxLength", ("minLength", "maxLength")),
    ("正则 pattern", ("pattern",)),
    ("数组大小 minItems/maxItems/uniqueItems", ("minItems", "maxItems", "uniqueItems")),
    (
        "数值范围 minimum/maximum/multipleOf",
        ("minimum", "maximum", "exclusiveMinimum", "exclusiveMaximum", "multipleOf"),
    ),
    ("格式 format", ("format",)),
)
_SC_ALL_CONSTRAINTS = tuple(k for _, keys in _SC_CONSTRAINT_CLASSES for k in keys)
_SC_ANNOTATIONS = ("title", "default", "description", "examples")
_SC_FIELDS = ("temperature", "max_completion_tokens", "stream", "seed")


class _ScHalt(Exception):
    """连接类异常 / 认证等非请求体问题：不再继续发请求。"""


def _sc_tiny_png():
    def chunk(kind, data):
        crc = (zlib.crc32(kind + data) & 0xFFFFFFFF).to_bytes(4, "big")
        return len(data).to_bytes(4, "big") + kind + data + crc

    header = (1).to_bytes(4, "big") * 2 + bytes([8, 2, 0, 0, 0])
    return (
        b"\x89PNG\r\n\x1a\n"
        + chunk(b"IHDR", header)
        + chunk(b"IDAT", zlib.compress(b"\x00\xff\xff\xff"))
        + chunk(b"IEND", b"")
    )


def _sc_capture_bodies(cfg, answer_seed):
    """用管线自己的 client 构造四种真实请求体；sender 只记下 payload 然后抛错，绝不联网。"""
    from enterprise_pdf_rag.adapters import document_tree_extraction as tree_module
    from enterprise_pdf_rag.adapters.http.layout_schemas import PageLayoutDTO
    from enterprise_pdf_rag.adapters.page_metadata_extraction import PageMetadataDTO
    from enterprise_pdf_rag.answers.prompt import SYSTEM_RULES, ModelAnswer

    held = {}

    def sender(url, *, api_key, payload, timeout):
        held["payload"] = payload
        raise ProviderRequestError("captured, not sent")

    with TemporaryDirectory() as cache:  # claim / context 文件只会落在这里，退出即删

        def build(call, seed):
            client = JsonCompletionClient(
                cfg,
                cache_dir=Path(cache),
                max_live_calls=1,
                timeout=_SC_TIMEOUT_S,
                sender=sender,
                seed=seed,
            )
            held.clear()
            try:
                call(client)
            except Exception:  # noqa: BLE001 - 预期：sender 抛出的占位异常
                pass
            return json.loads(held["payload"])

        text_prompt = (
            'Physical page: 1.\nSource text spans:\n[{"id":"s0000","text":"Revenue 2024 HK$1"}]'
        )
        return {
            "text": build(
                lambda c: c.complete_text_json(
                    task="selfcheck-page-metadata",
                    prompt=text_prompt,
                    response_model=PageMetadataDTO,
                    max_output_tokens=1024,
                ),
                None,
            ),
            "vision": build(
                lambda c: c.complete_json(
                    task="selfcheck-page-layout",
                    prompt="Physical page: 1.\nSource text observations:\n[]",
                    image_png=_sc_tiny_png(),
                    response_model=PageLayoutDTO,
                    max_output_tokens=4096,
                ),
                None,
            ),
            "tree": build(
                lambda c: c.complete_text_json(
                    task="selfcheck-tree-summary",
                    prompt="Section: Overview, pages p1-p2.\nPage text: Revenue 2024 HK$1",
                    response_model=tree_module.TreeSummaryDTO,
                    system=getattr(tree_module, "_SUMMARY_RULES", None),
                    max_output_tokens=1024,
                ),
                None,
            ),
            "answer": build(
                lambda c: c.complete_text_json(
                    task="selfcheck-answer",
                    prompt="Question: What was revenue?\nContext: [c1] Revenue 2024 HK$1",
                    response_model=ModelAnswer,
                    system=SYSTEM_RULES,
                ),
                answer_seed,
            ),
        }


# ── schema 变换：只在 schema 节点上动手，properties 里的字段名不会被误删 ──
def _sc_walk(node, fn):
    if not isinstance(node, dict):
        return node
    out = {}
    for key, value in node.items():
        if key in ("properties", "$defs", "definitions") and isinstance(value, dict):
            out[key] = {name: _sc_walk(sub, fn) for name, sub in value.items()}
        elif key in ("anyOf", "oneOf", "allOf", "prefixItems") and isinstance(value, list):
            out[key] = [_sc_walk(sub, fn) for sub in value]
        elif key in ("items", "additionalProperties", "not") and isinstance(value, dict):
            out[key] = _sc_walk(value, fn)
        else:
            out[key] = value
    return fn(out)


def _sc_schema_of(body):
    return body["response_format"]["json_schema"]["schema"]


def _sc_with_schema(body, change):
    out = copy.deepcopy(body)
    out["response_format"]["json_schema"]["schema"] = change(_sc_schema_of(out))
    return out


def _sc_strip_keys(keys):
    def change(schema):
        return _sc_walk(schema, lambda n: {k: v for k, v in n.items() if k not in keys})

    return lambda body: _sc_with_schema(body, change)


def _sc_inline_refs(schema):
    defs = schema.get("$defs") or {}

    def expand(node, stack):
        def fn(n):
            ref = n.get("$ref")
            name = ref.rsplit("/", 1)[-1] if isinstance(ref, str) else None
            if name in defs and name not in stack:
                rest = {k: v for k, v in n.items() if k != "$ref"}
                return {**expand(defs[name], stack + (name,)), **rest}
            return n

        return _sc_walk(node, fn)

    return expand({k: v for k, v in schema.items() if k != "$defs"}, ())


def _sc_drop_any_of(schema):
    def fn(n):
        branches = n.get("anyOf")
        if not isinstance(branches, list) or not branches:
            return n
        real = [b for b in branches if isinstance(b, dict) and b.get("type") != "null"]
        pick = (real or branches)[0]
        return {**{k: v for k, v in n.items() if k != "anyOf"}, **pick}

    return _sc_walk(schema, fn)


def _sc_set_strict_false(body):
    out = copy.deepcopy(body)
    out["response_format"]["json_schema"]["strict"] = False
    return out


def _sc_without(*names):
    return lambda body: {k: v for k, v in body.items() if k not in names}


def _sc_json_object(body):
    return {**body, "response_format": {"type": "json_object"}}


def _sc_compose(*changes):
    def apply(body):
        for change in changes:
            body = change(body)
        return body

    return apply


def _sc_keys_present(schema):
    found = set()

    def fn(n):
        found.update(n)
        return n

    _sc_walk(schema, fn)
    return found


def _sc_text_only(body):
    out = copy.deepcopy(body)
    for message in out["messages"]:
        if isinstance(message["content"], list):
            message["content"] = "".join(
                part.get("text", "") for part in message["content"] if part.get("type") == "text"
            )
    return out


def _sc_ladder(body):
    """response_format 逐级放宽（累计）：[(被放宽掉的特性, 变换)]；schema 里没有的特性直接略过，不发请求。"""
    present = _sc_keys_present(_sc_schema_of(body))
    steps = []
    if present & set(_SC_ALL_CONSTRAINTS):
        steps.append(
            (
                "schema 约束关键字（minLength / maxLength / pattern / minItems / maxItems / format 等）",
                _sc_strip_keys(_SC_ALL_CONSTRAINTS),
            )
        )
    if present & {"$defs", "$ref"}:
        steps.append(
            ("$defs / $ref 引用（改为内联展开）", lambda b: _sc_with_schema(b, _sc_inline_refs))
        )
    if "anyOf" in present:
        steps.append(("anyOf（可空 / 联合类型）", lambda b: _sc_with_schema(b, _sc_drop_any_of)))
    if present & set(_SC_ANNOTATIONS):
        steps.append(
            ("title / default / description 等注解关键字", _sc_strip_keys(_SC_ANNOTATIONS))
        )
    steps.append(("strict: true（改为 strict: false）", _sc_set_strict_false))
    steps.append(('json_schema 整体（改为 {"type": "json_object"}）', _sc_json_object))
    return steps


def _sc_restore_only(keys):
    """去掉全部约束关键字，只把 keys 这一类留回去。"""
    return _sc_strip_keys(tuple(k for k in _SC_ALL_CONSTRAINTS if k not in keys))


def run_llm_selfcheck():
    """返回要抛给主运行的中文说明（需要阻断时），否则返回 None；所有诊断先打印完。"""
    try:
        cfg = load_llm_config()
    except ProviderConfigurationError as error:
        print("LLM 配置未就绪，跳过自检（主运行的 preflight 会指出缺哪些变量）：", error)
        return None
    auth = cfg.api_key.get_secret_value()
    parts = urlsplit(cfg.chat_completions_url)

    def clean(text, limit=_SC_BODY_CHARS):
        return " ".join(str(text).replace(auth, "***").split())[:limit]

    print("目标 URL :", f"{parts.scheme}://{parts.netloc}{parts.path}")
    print("模型名   :", cfg.model)
    print(
        f"每个请求超时 {_SC_TIMEOUT_S:.0f} 秒；只发固定的极小合成文本；不写磁盘、不改环境变量；不打印 key"
    )

    def post(body):
        started = time.monotonic()
        connection = HTTPSConnection(parts.netloc, timeout=_SC_TIMEOUT_S)
        try:
            connection.request(
                "POST",
                parts.path,
                body=json.dumps(body, sort_keys=True, separators=(",", ":")).encode(),
                headers={"Authorization": "Bearer " + auth, "Content-Type": "application/json"},
            )
            response = connection.getresponse()
            raw = response.read(4096).decode("utf-8", "replace")
            return response.status, time.monotonic() - started, raw
        except (OSError, HTTPException) as error:
            return None, time.monotonic() - started, f"{type(error).__name__}: {error}"
        finally:
            connection.close()

    sent = [0]
    halt = {}

    def probe(label, body):
        """发 1 次并打印一行；返回 (是否通过, 状态码, 服务端正文)。连接类 / 非请求体问题抛 _ScHalt。"""
        sent[0] += 1
        status, elapsed, raw = post(body)
        if status is None:
            print(f"  [连接失败] {label} | {elapsed:.2f}s | {clean(raw)}")
            halt.update(kind="net", detail=clean(raw))
            raise _ScHalt
        if 200 <= status < 300:
            print(f"  [通过] {label} | {elapsed:.2f}s")
            return True, status, raw
        print(f"  [被拒 HTTP {status}] {label} | {elapsed:.2f}s | 服务端: {clean(raw)}")
        if status in (401, 403, 404, 407, 408, 429) or status >= 500:
            halt.update(kind="env", detail=f"HTTP {status} {clean(raw)}")
            raise _ScHalt
        return False, status, raw

    def narrow_constraints(body, combined):
        """去掉全部约束关键字能通过时：每次只恢复一类，指出具体是哪一类。"""
        present = _sc_keys_present(_sc_schema_of(body))
        for klass, keys in _SC_CONSTRAINT_CLASSES:
            if not present & set(keys):
                continue
            ok, _, _ = probe(f"去掉全部约束关键字，只恢复「{klass}」", _sc_restore_only(keys)(body))
            if not ok:
                return {
                    "culprit": f"schema 约束关键字中的「{klass}」",
                    "accepted": "response_format 保留 json_schema，但去掉了该类约束关键字",
                    "relax": _sc_strip_keys(keys),
                }
        return {
            "culprit": "schema 约束关键字（单类都能通过，需多类组合才被拒）",
            "accepted": "response_format 保留 json_schema，但去掉了全部约束关键字",
            "relax": combined,
        }

    def bisect_response_format(body):
        done = []
        for feature, change in _sc_ladder(body):
            done.append((feature, change))
            combined = _sc_compose(*[c for _, c in done])
            ok, _, _ = probe("response_format 累计放宽，去掉 " + feature, combined(body))
            if not ok:
                continue
            if feature.startswith("schema 约束"):
                return narrow_constraints(body, combined)
            if feature.startswith("json_schema 整体"):
                return {
                    "culprit": "response_format 的 json_schema 整体不被接受（json_object 可用）",
                    "accepted": 'response_format 改为 {"type": "json_object"}',
                    "relax": combined,
                }
            return {
                "culprit": feature,
                "accepted": "response_format 保留 json_schema，累计去掉："
                + "；".join(f for f, _ in done),
                "relax": combined,
            }
        return {
            "culprit": "response_format 本身：json_schema 与 json_object 都不被接受",
            "accepted": "不带 response_format 的请求（其余字段原样）",
            "relax": _sc_without("response_format"),
        }

    def bisect(body, vision):
        """定位一个被拒形态的原因，返回 dict(culprit, accepted, relax)；relax 是可复用的修复变换。"""
        ok, _, _ = probe("去掉 response_format", _sc_without("response_format")(body))
        if ok:
            return bisect_response_format(body)
        minimal = {k: body[k] for k in ("model", "messages")}
        ok, _, _ = probe("最小请求（只留 model + messages）", minimal)
        if ok:
            for name in _SC_FIELDS:
                if name not in body:
                    continue
                ok, _, _ = probe(f"最小请求 + {name}", {**minimal, name: body[name]})
                if ok:
                    continue
                note = ""
                if name == "max_completion_tokens":
                    works, _, _ = probe(
                        "最小请求 + max_tokens（改名）", {**minimal, "max_tokens": body[name]}
                    )
                    note = "（改用 max_tokens 可通过）" if works else "（改用 max_tokens 也不行）"
                return {
                    "culprit": f"请求字段 {name}{note}",
                    "accepted": f"最小请求（model + messages），单独加回 {name} 即被拒",
                    "relax": _sc_without(name),
                    "field": name,
                }
            return {
                "culprit": "单个字段都能通过，temperature / max_completion_tokens / stream / seed 组合起来才被拒",
                "accepted": "最小请求（model + messages）及逐个加回的单字段请求",
                "relax": None,
            }
        if vision:
            ok, _, _ = probe("最小请求，图片改为纯文本", _sc_text_only(minimal))
            if ok:
                return {
                    "culprit": "messages 里的 image_url（图片输入）",
                    "accepted": "不含图片的最小请求（model + messages 纯文本）",
                    "relax": None,
                }
        return {
            "culprit": "连最小请求（只有 model + messages）都被拒：问题不在请求体字段，请看服务端原话（URL / 认证 / 模型名）",
            "accepted": "（没有找到被接受的形态）",
            "relax": None,
        }

    forms = (
        ("text", "入库 text（页元数据）", True),
        ("vision", "入库 vision（版面，含图片）", True),
        ("tree", "tree 建树（节点摘要）", False),
        ("answer", "回答阶段（带 seed）", False),
    )
    try:
        bodies = _sc_capture_bodies(cfg, get_settings().answer_seed)
    except Exception as error:  # noqa: BLE001 - 取不到请求体是自检自己的问题，交给外层处理
        raise RuntimeError("无法构造管线的请求体：" + clean(error)) from None

    results = {}
    primary = None
    try:
        print("\n== 1. 管线真实请求体的原样基线（各形态 1 次）==")
        for key, label, _ in forms:
            ok, status, raw = probe(label + " 原样请求", bodies[key])
            results[key] = {"ok": ok, "status": status, "server": clean(raw)}
        rejected = [key for key, _, _ in forms if not results[key]["ok"]]
        if rejected:
            print("\n== 2. 二分定位（只对被拒的形态；后面的形态先试同一修复，通过就不再展开）==")
        reusable = None
        for key, label, _ in forms:
            if key not in rejected:
                continue
            if reusable is not None:
                ok, _, _ = probe(f"{label}：套用已定位的修复", reusable["relax"](bodies[key]))
                if ok:
                    results[key]["finding"] = {"culprit": "与第一个被拒形态同一原因（去掉后通过）"}
                    results[key]["fixed"] = True
                    continue
            print(f"  -- 定位：{label} --")
            finding = bisect(bodies[key], key == "vision")
            results[key]["finding"] = finding
            if finding.get("field") in DEGRADABLE_SAMPLING_PARAMETERS:
                ok, _, _ = probe(f"{label}：原样请求只去掉 {finding['field']}", finding["relax"](bodies[key]))
                results[key]["fixed"] = ok
            primary = primary or key
            if reusable is None and finding["relax"] is not None:
                reusable = finding
    except _ScHalt:
        pass

    ingest_rejected = [
        label
        for key, label, ingest in forms
        if ingest and key in results and not results[key]["ok"]
    ]
    block = None
    print("\n" + "=" * 12 + " 结论 " + "=" * 12)
    if halt.get("kind") == "net":
        print(
            "结论: 连接失败，网络 / 代理 / 证书问题，不是请求体问题；不再做字段二分，将阻断主运行"
        )
        print("  请检查 OPENAI_BASE_URL、代理、证书、DNS；异常:", halt["detail"])
        block = "连接失败（网络 / 代理 / 证书问题）"
    elif halt:
        print(
            f"结论: 端点返回 {halt['detail'][:80]}，这不是请求体问题（认证 / 地址 / 限流 / 服务端）；不再做字段二分，将阻断主运行"
        )
        print("  请检查 key、OPENAI_BASE_URL（含路径）、模型名、权限与限流")
        block = "端点拒绝了请求，原因不在请求体（认证 / 地址 / 限流 / 服务端）"
    elif primary is None:
        print(
            f"结论: 四种请求形态（入库 text / vision、tree、回答）都被端点接受，400 不是请求体问题；共发 {sent[0]} 次请求，不阻断主运行"
        )
    else:
        finding = results[primary]["finding"]
        label = {k: v for k, v, _ in forms}[primary]
        field = finding.get("field")
        auto = field in DEGRADABLE_SAMPLING_PARAMETERS and all(
            results[key].get("fixed") for key, _, _ in forms if not results[key]["ok"]
        )
        if auto:
            switch = (
                "在 .env 里设 OPENAI_TEMPERATURE=omit"
                if field == "temperature"
                else "在 settings.yaml 里设 answer_seed: null"
            )
            print(
                f"结论: 端点不接受 {field}；管线会自动去掉它重发，不阻断主运行"
                f"（或{switch}，直接关闭、省掉每次运行的一次被拒探测）"
            )
        elif ingest_rejected:
            block = "入库阶段请求被端点拒绝：" + "、".join(ingest_rejected)
            verdict = "入库形态被拒，将阻断主运行"
        else:
            verdict = "入库形态均通过，只警告、不阻断主运行"
        if not auto:
            print(f"结论: 请求体里有被端点拒绝的内容（按「{label}」定位）；{verdict}")
        print("  最先被拒的一项 :", finding["culprit"])
        print("  最接近原样且仍被接受的形态 :", finding["accepted"])
        print("  服务端原话 :", results[primary]["server"])
        print("  各形态:")
        for key, label, _ in forms:
            if results[key]["ok"]:
                print(f"    [通过] {label}")
            else:
                why = "见上面的结论" if key == primary else results[key]["finding"]["culprit"]
                print(f"    [被拒 HTTP {results[key]['status']}] {label}：{why}")
    print(f"共发出 {sent[0]} 次请求")
    if block:
        return (
            f"LLM 自检未通过（{block}）。继续运行主流程只会让每个 PDF 的每次模型调用都失败，"
            "最终得到 status=nothing_to_index，并把失败结果永久缓存到 data/ingestion 下的 model-cache；"
            "已在此处停止，尚未写入任何入库缓存。请把本格输出整段发回；"
            "若确认要跳过自检，把配置格里的 LLM_SELFCHECK 设为 False。"
        )
    return None


_sc_block = None
if not LLM_SELFCHECK:
    print("LLM_SELFCHECK = False：跳过 LLM 请求自检")
else:
    try:
        _sc_block = run_llm_selfcheck()
    except Exception as _sc_error:  # noqa: BLE001 - 自检自身的 bug 不得阻断主运行
        print(
            "自检代码自身出错（不影响主运行，继续）：",
            type(_sc_error).__name__,
            str(_sc_error)[:300],
        )
if _sc_block:
    raise RuntimeError(_sc_block)

In [ ]:
from enterprise_pdf_rag.adapters.folder_pipeline import run_folder_pipeline


def progress(event, payload):
    print(event, payload)


result = run_folder_pipeline(
    PDF_DIR,  # None → 取 NB_PDF_DIR；两处都没有会直接报错
    questions=QUESTIONS,
    ingestion_root=INGESTION_ROOT,
    max_live_calls_per_pdf=MAX_LIVE_CALLS_PER_PDF,
    max_live_calls_total=MAX_LIVE_CALLS_TOTAL,
    max_questions=MAX_QUESTIONS,  # 只回答前 N 道题；None = 全部
    build_tree=BUILD_TREE,
    requalify=REQUALIFY,
    report_dir=REPORT_DIR,
    progress=progress,
)

In [ ]:
from pathlib import Path


def show(rows, columns):
    """不依赖 pandas 的等宽表格。"""
    cells = [[str(row.get(column, "")) for column in columns] for row in rows]
    widths = [
        max([len(column), *(len(line[i]) for line in cells)]) for i, column in enumerate(columns)
    ]
    print("  ".join(column.ljust(width) for column, width in zip(columns, widths)))
    for line in cells:
        print("  ".join(value.ljust(width) for value, width in zip(line, widths)))


print("ok              :", result.ok)
print("live_calls      :", result.live_calls.model_dump())
print("budget_exhausted:", result.budget_exhausted)
print("report_dir      :", result.report_dir)

documents = [
    {
        "pdf": Path(item.pdf_path).name,
        "status": item.status,
        "live_calls": item.live_calls,
        "elapsed_s": round(item.elapsed_s, 1),
        # 图表是否进了索引：qualification.chart_member_count > 0
        "chart_member_count": None
        if item.qualification is None
        else item.qualification.chart_member_count,
        "indexed_members": None if item.publication is None else item.publication.member_count,
        "error": item.error or "",
    }
    for item in result.documents
]
show(
    documents,
    ["pdf", "status", "live_calls", "elapsed_s", "chart_member_count", "indexed_members", "error"],
)

In [ ]:
if result.eval is None:
    print("没有题集（NB_QUESTIONS_PATH 未设置）：只做了入库。")
else:
    if MAX_QUESTIONS is not None:
        print(f"本次只回答题集的前 {MAX_QUESTIONS} 道题（MAX_QUESTIONS；改成 None 回答全部），实际回答 {len(result.eval.cases)} 道")
    print("totals:", result.eval.totals)
    show(
        [
            {
                "case": case.case_id,
                "verdict": case.verdict,
                "status": case.status or "",
                "claims": case.claim_count,
                "cited_pages": list(case.cited_pages),
                "page_rank": case.page_rank,
                "ms": round(case.elapsed_ms),
                "failures": "; ".join(case.failures)[:80],
            }
            for case in result.eval.cases
        ],
        ["case", "verdict", "status", "claims", "cited_pages", "page_rank", "ms", "failures"],
    )

In [ ]:
# 结果文件：只保留 question / expected / answer 三列，整文件一次写到 REPORT_DIR / "answers.csv"
# （UTF-8 带 BOM，Excel 直接打开不乱码；先写同目录临时名再 os.replace，不追加写）
import csv
import io
import os

if result.eval is None or not result.eval.cases:
    print("没有题集（NB_QUESTIONS_PATH 未设置）或没有任何用例：不生成 answers.csv")
else:
    ANSWERS_CSV = REPORT_DIR / "answers.csv"
    _buffer = io.StringIO(newline="")
    _writer = csv.writer(_buffer)
    _writer.writerow(["question", "expected", "answer"])
    for _case in result.eval.cases:  # 与题集顺序一致，答题失败的题也占一行（answer 为空）
        _writer.writerow([_case.question, _case.expected or "", _case.answer or ""])
    ANSWERS_CSV.parent.mkdir(parents=True, exist_ok=True)
    _partial = ANSWERS_CSV.with_name(ANSWERS_CSV.name + ".partial")
    _partial.write_bytes(_buffer.getvalue().encode("utf-8-sig"))
    os.replace(_partial, ANSWERS_CSV)
    print("已写出", ANSWERS_CSV, "共", len(result.eval.cases), "题")
    for _case in result.eval.cases:
        print(
            "-",
            _case.case_id,
            "|",
            (_case.expected or "").replace("\n", " ")[:40],
            "|",
            (_case.answer or "").replace("\n", " ")[:80],
        )

## 可选：单独拿一个回答模型客户端

`make_answer_llm()` 用 `.env` 里的 LLM 配置（`OPENAI_*`，别名 `APP_LLM_*`）构造 `JsonCompletionClient`，
缓存目录、调用预算、超时与种子取自 settings。构造本身不发请求；配置缺失时在这里抛 `ProviderConfigurationError`。

In [ ]:
from enterprise_pdf_rag.adapters.answer_llm import make_answer_llm

llm = make_answer_llm()  # 或 make_answer_llm(cache_dir=..., max_live_calls=...)
print(type(llm).__name__, "live calls so far:", llm.live_call_count)